# Medical AI Evaluation: Beyond Accuracy

## 1. The Accuracy Fallacy
In a dataset where 90% of samples are harmless moles (nevi) and 10% are deadly melanoma, a model that *always* predicts "harmless" has **90% accuracy**.
**However, it kills 10% of patients.**

Thus, accuracy is a misleading metric in medical statistics. We need metrics that punish false negatives (missing a cancer).

## 2. Key Metrics: Confusion Matrix Definitions

Let **Positive** = Melanoma (Disease), **Negative** = Benign (Healthy).
- **TP (True Positive):** Correctly spotted melanoma.
- **TN (True Negative):** Correctly identified benign mole.
- **FP (False Positive - Type I Error):** "False Alarm". Benign mole called cancer. Causes anxiety/biopsy but patient lives.
- **FN (False Negative - Type II Error):** "Missed Diagnosis". Cancer called benign. Patient is sent home and may die.

### 2.1. Sensitivity (Recall)
$$ Sensitivity = \frac{TP}{TP + FN} $$
Given a patient HAS cancer, how likely are we to detect it? In our project, we want this near 100%.

### 2.2. Specificity
$$ Specificity = \frac{TN}{TN + FP} $$
Given a patient is HEALTHY, how likely are we to say so? Low specificity means too many unnecessary biopsies.

## 3. Class Weights: The Mathematical Fix

To solve the imbalance, we use **Weighted Cross-Entropy Loss**.
Instead of every image counting as "1 point" of error, we assign higher points to rare classes.

$$ Weight_{class} = \frac{Total Samples}{Num Classes \times Count_{class}} $$

In `src.skin_cancer_detection.data`, we calculate this automatically:

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
from sklearn.utils.class_weight import compute_class_weight

# Mock data to demonstrate the concept without loading full dataset
# Assume 10 Melanoma (rare) and 90 Nevi (common)
labels = ['mel'] * 10 + ['nv'] * 90
classes = np.unique(labels)

weights = compute_class_weight(class_weight='balanced', classes=classes, y=labels)
weight_dict = dict(zip(classes, weights))

print("Class Weights:")
for cls, w in weight_dict.items():
    print(f"  {cls}: {w:.4f}")

**Result:** You will see 'mel' has a much higher weight (e.g., 5.0) than 'nv' (e.g., 0.55). The model is "paid" 10x more for getting a melanoma right than a nevi right.

## 4. ROC Curve and AUC

The Receiver Operating Characteristic (ROC) curve plots **Sensitivity (y-axis)** vs **1 - Specificity (x-axis)** at different threshold settings.

- **AUC = 0.5:** Random guessing.
- **AUC = 1.0:** Perfect classifier.
- **AUC = 0.8+:** Generally considered good for screening.

We use AUC as a primary metric during training to ensure the model is robust regardless of the specific probability threshold chosen.